# Class 11 — Advanced Classification Models
## Hands-On Lab: Naive Bayes, SVM & KNN




In [ ]:
# ============================================================
# CELL 1: Environment Setup & Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# Visual settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ All libraries imported successfully!")


## Part 1: Naive Bayes — The Probabilistic Classifier 

### Learning Objectives 

* **Understand Bayes' Theorem in practice** — প্র্যাক্টিক্যাল লাইফে বা বাস্তব উদাহরণের মাধ্যমে বেইজ থিওরেম (Bayes' Theorem) কীভাবে কাজ করে এবং এর গাণিতিক ভিত্তি গভীরভাবে বোঝা।
* **See why the "naive" independence assumption works for text** — টেক্সট ক্লাসিফিকেশন বা ন্যাচারাল ল্যাঙ্গুয়েজ প্রসেসিংয়ের (NLP) ক্ষেত্রে সব ফিচারকে স্বাধীন বা "Naive" ধরে নেওয়ার যে অদ্ভুত অনুমান (Independence assumption), তা ভুল হওয়া সত্ত্বেও বাস্তবে কেন এত নিখুঁতভাবে কাজ করে তা বিশ্লেষণ করা।
* **Build a spam classifier from scratch intuition** — কোনো জটিল লাইব্রেরি ছাড়াই একদম স্ক্র্যাচ (Scratch) বা শূন্য থেকে একটি রিয়েল-ওয়ার্ল্ড স্প্যাম ইমেইল ফিল্টারিং সিস্টেম (Spam Classifier) তৈরির মূল থট প্রসেস ও মেকানিজম আয়ত্ত করা।


In [ ]:
# ============================================================
# CELL 2: Naive Bayes — Synthetic Dataset Demo
# ============================================================

# Create a simple 2D dataset where classes are somewhat separable
X_nb, y_nb = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    random_state=42
)

# Split
X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_nb, y_nb, test_size=0.3, random_state=42
)

# Scale for GaussianNB (it assumes normal distribution)
scaler_nb = StandardScaler()
X_train_nb_s = scaler_nb.fit_transform(X_train_nb)
X_test_nb_s = scaler_nb.transform(X_test_nb)

# Train Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train_nb_s, y_train_nb)
y_pred_nb = gnb.predict(X_test_nb_s)

# Metrics
print("=== NAIVE BAYES RESULTS ===")
print(f"Accuracy:  {accuracy_score(y_test_nb, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test_nb, y_pred_nb):.4f}")
print(f"Recall:    {recall_score(y_test_nb, y_pred_nb):.4f}")
print(f"F1 Score:  {f1_score(y_test_nb, y_pred_nb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_nb, y_pred_nb))

# Plot decision boundary
h = 0.02
x_min, x_max = X_test_nb_s[:, 0].min() - 1, X_test_nb_s[:, 0].max() + 1
y_min, y_max = X_test_nb_s[:, 1].min() - 1, X_test_nb_s[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = gnb.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
scatter = plt.scatter(X_test_nb_s[:, 0], X_test_nb_s[:, 1], c=y_test_nb, cmap=plt.cm.RdYlBu, edgecolors='k')
plt.title('Naive Bayes Decision Boundary (GaussianNB)', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.colorbar(scatter, label='Class')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 3: Naive Bayes — Text Classification (Spam Detection)
# ============================================================

# Synthetic text data for demonstration
spam_texts = [
    "Congratulations you won a free prize call now",
    "URGENT you have won a cash reward claim immediately",
    "Buy cheap pills now click here limited offer",
    "You are selected for a gift card act fast",
    "Free entry win a car call this number",
    "Act now limited time offer buy one get one",
    "You have a refund pending confirm bank details",
    "Call now to claim your prize money waiting",
    "Free vacation package claim your reward today",
    "Urgent your account has been compromised reset password",
]

ham_texts = [
    "Hey are we still on for lunch tomorrow",
    "Can you send me the notes from class",
    "The meeting is rescheduled to 3 PM",
    "Happy birthday hope you have a great day",
    "Please find the attached report for review",
    "Thanks for the help with the project",
    "Don't forget to pick up milk on your way home",
    "The package has been delivered tracking updated",
    "See you at the gym after work today",
    "Reminder dentist appointment at 10 AM tomorrow",
]

# Create DataFrame
texts = spam_texts + ham_texts
labels = ['spam'] * len(spam_texts) + ['ham'] * len(ham_texts)
df_text = pd.DataFrame({'text': texts, 'label': labels})

# Shuffle
df_text = df_text.sample(frac=1, random_state=42).reset_index(drop=True)

# Vectorize with CountVectorizer (word counts — classic for Naive Bayes)
vectorizer_nb = CountVectorizer()
X_text = vectorizer_nb.fit_transform(df_text['text'])
y_text = df_text['label']

# Split
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42, stratify=y_text
)

# Train Multinomial Naive Bayes (the standard for text)
mnb = MultinomialNB(alpha=1.0)  # alpha = Laplace smoothing
mnb.fit(X_train_txt, y_train_txt)
y_pred_txt = mnb.predict(X_test_txt)

# Show learned probabilities for a few words
feature_names = vectorizer_nb.get_feature_names_out()
log_probs = mnb.feature_log_prob_

print("=== MULTINOMIAL NAIVE BAYES — TEXT CLASSIFICATION ===")
print(f"Accuracy:  {accuracy_score(y_test_txt, y_pred_txt):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test_txt, y_pred_txt, labels=['ham', 'spam']))

# Show top spam-indicating words
top_spam_indices = np.argsort(log_probs[1])[-10:]  # class 1 = spam
print("\nTop 10 words indicating SPAM:")
for idx in reversed(top_spam_indices):
    print(f"  {feature_names[idx]:15s} | log-prob: {log_probs[1][idx]:.4f}")

# Show top ham-indicating words
top_ham_indices = np.argsort(log_probs[0])[-10:]  # class 0 = ham
print("\nTop 10 words indicating HAM:")
for idx in reversed(top_ham_indices):
    print(f"  {feature_names[idx]:15s} | log-prob: {log_probs[0][idx]:.4f}")


## Part 2: Support Vector Machines (SVM) 

### Learning Objectives 

* **Understand the concept of margin maximization** — দুটি ক্লাসের মধ্যে সর্বোচ্চ দূরত্ব বা মার্জিন তৈরি করে কীভাবে একটি অপ্টিমাল বা সেরা বাউন্ডারি (Maximum Margin Hyperplane) টানা যায়, সেই জ্যামিতিক কনসেপ্টটি গভীরভাবে বোঝা।
* **See the kernel trick in action** — লিনিয়ারলি নন-সেপারেবল ডেটাকে (যা সোজা লাইনে ভাগ করা যায় না) কার্নেল ট্রিক (Kernel Trick) ব্যবহার করে কীভাবে উচ্চ মাত্রার স্পেসে (Higher-dimensional space) নিয়ে গিয়ে খুব সহজে আলাদা করা যায়, তা প্র্যাক্টিক্যাল ফিল দিয়ে দেখা।
* **Compare Linear vs. RBF kernels** — সিম্পল ডেটার জন্য লিনিয়ার কার্নেল (Linear Kernel) এবং জটিল বা সার্কুলার ডেটার জন্য আরবিএফ কার্নেল (RBF Kernel) — এই দুটির কার্যকারিতা, পার্থক্য এবং ব্যবহারের সঠিক ক্ষেত্রগুলো তুলনা করা।
* **Build an SVM classifier with hyperparameter tuning** — একটি রিয়েল-ওয়ার্ল্ড SVM Classifier মডেল বিল্ড করা এবং গ্রিড সার্চ (GridSearchCV) ব্যবহার করে এর মূল হাইপারপ্যারামিটারগুলো ($C$ এবং $\gamma$ - Gamma) নিখুঁতভাবে টিউন বা অপ্টিমাইজ করা শেখা।


In [ ]:
# ============================================================
# CELL 4: SVM — Linear Kernel on Linearly Separable Data
# ============================================================

# Create linearly separable data
X_svm, y_svm = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.5,  # well-separated classes
    random_state=42
)

X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm, test_size=0.3, random_state=42
)

# Train Linear SVM
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_svm, y_train_svm)
y_pred_svm = svm_linear.predict(X_test_svm)

print("=== SVM — LINEAR KERNEL ===")
print(f"Accuracy:  {accuracy_score(y_test_svm, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test_svm, y_pred_svm):.4f}")
print(f"Recall:    {recall_score(y_test_svm, y_pred_svm):.4f}")
print(f"F1 Score:  {f1_score(y_test_svm, y_pred_svm):.4f}")
print(f"\nNumber of Support Vectors: {svm_linear.n_support_.sum()}")
print(f"Support Vectors per class: {svm_linear.n_support_}")

# Plot decision boundary + margin
h = 0.02
x_min, x_max = X_train_svm[:, 0].min() - 1, X_train_svm[:, 0].max() + 1
y_min, y_max = X_train_svm[:, 1].min() - 1, X_train_svm[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_linear.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(12, 5))

# Plot 1: Decision boundary
plt.subplot(1, 2, 1)
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
plt.scatter(X_train_svm[:, 0], X_train_svm[:, 1], c=y_train_svm, cmap=plt.cm.coolwarm, edgecolors='k', s=50)
# Plot support vectors
plt.scatter(svm_linear.support_vectors_[:, 0], svm_linear.support_vectors_[:, 1], 
            s=150, facecolors='none', edgecolors='black', linewidths=1.5, label='Support Vectors')
plt.title('SVM Linear Kernel — Decision Boundary', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()

# Plot 2: Margin visualization (decision function)
plt.subplot(1, 2, 2)
Z_dist = svm_linear.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, Z_dist, levels=20, alpha=0.6, cmap=plt.cm.RdBu)
plt.scatter(X_train_svm[:, 0], X_train_svm[:, 1], c=y_train_svm, cmap=plt.cm.RdBu, edgecolors='k', s=50)
plt.contour(xx, yy, Z_dist, levels=[-1, 0, 1], linestyles=['--', '-', '--'], colors='k', linewidths=1.5)
plt.title('SVM Margin Visualization', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(label='Distance from Hyperplane')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 5: SVM — The Kernel Trick on Non-Linear Data
# ============================================================

# Create moon-shaped (non-linearly separable) data
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

# Compare Linear vs RBF kernel
kernels = ['linear', 'rbf']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, kernel in enumerate(kernels):
    svm_k = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    svm_k.fit(X_train_m, y_train_m)
    y_pred_m = svm_k.predict(X_test_m)

    acc = accuracy_score(y_test_m, y_pred_m)

    # Decision boundary
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.coolwarm)
    axes[idx].scatter(X_train_m[:, 0], X_train_m[:, 1], c=y_train_m, cmap=plt.cm.coolwarm, edgecolors='k', s=50)
    axes[idx].set_title(f'SVM with {kernel.upper()} Kernel\nAccuracy: {acc:.4f}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.suptitle('The Kernel Trick: Linear vs. RBF on Moon-Shaped Data', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n=== OBSERVATION ===")
print("Linear kernel fails on non-linear data. RBF kernel creates a curved boundary that fits the moon shapes.")


In [ ]:
# ============================================================
# CELL 6: SVM — Hyperparameter Tuning (C and Gamma)
# ============================================================

# Grid search over C and gamma for RBF kernel
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

grid_svm = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train_m, y_train_m)

print("=== SVM GRID SEARCH RESULTS ===")
print(f"Best Parameters: {grid_svm.best_params_}")
print(f"Best CV Accuracy: {grid_svm.best_score_:.4f}")
print(f"Test Accuracy:    {accuracy_score(y_test_m, grid_svm.predict(X_test_m)):.4f}")

# Visualize C effect
C_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, len(C_values), figsize=(18, 3))

for idx, C in enumerate(C_values):
    svm_c = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    svm_c.fit(X_train_m, y_train_m)

    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm_c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.coolwarm)
    axes[idx].scatter(X_train_m[:, 0], X_train_m[:, 1], c=y_train_m, cmap=plt.cm.coolwarm, edgecolors='k', s=20)
    axes[idx].set_title(f'C = {C}', fontsize=11)
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])

plt.suptitle('Effect of Regularization Parameter C on Decision Boundary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== INTERPRETATION ===")
print("Low C  → Wider margin, more misclassification allowed (underfitting).")
print("High C → Narrow margin, tries to classify every point correctly (overfitting).")


## Part 3: K-Nearest Neighbors (KNN) 

### Learning Objectives 

* **Understand instance-based (lazy) learning** — মডেলের কোনো ট্রেইনিং ফেজ বা ওয়েট আপডেট না করে, সম্পূর্ণ ইনস্ট্যান্স-বেসড (Instance-based) বা অলস লার্নিং (Lazy learning) কীভাবে কাজ করে এবং রিয়েল-টাইমে এটি কেন সরাসরি মেমোরি থেকে ডেটা ম্যাচ করে তা গভীরভাবে বোঝা।
* **See how $k$ and distance metrics affect predictions** — প্রতিবেশীর সংখ্যা ($k$-এর মান) এবং দূরত্ব মাপার বিভিন্ন উপায় যেমন: ইউক্লিডিয়ান (Euclidean) বা ম্যানহাটন (Manhattan) ডিসট্যান্স কীভাবে ফাইনাল প্রেডিকশনকে সরাসরি প্রভাবিত করে তা গাণিতিকভাবে দেখা।
* **Visualize decision boundaries for different k values** — $k$-এর মান ছোট (যেমন: $k=1$) বা বড় (যেমন: $k=20$) করার কারণে মডেলের ডিসিশন বাউন্ডারি (Decision boundaries) কতটা স্মুথ বা আঁকাবাঁকা (Overfit vs Underfit) হচ্ছে, তা গ্রাফের মাধ্যমে ভিজ্যুয়ালাইজ করা।
* **Build a KNN classifier and compare distance weighting** — একটি রিয়েল-ওয়ার্ল্ড KNN Classifier মডেল তৈরি করা এবং সাধারণ মেজরিটি ভোট (Uniform weighting) বনাম দূরত্বের ওপর ভিত্তি করে ভোটের ওজন নির্ধারণের (Distance-based weighting) পারফরম্যান্স প্র্যাক্টিক্যালি তুলনা করা।


In [ ]:
# ============================================================
# CELL 7: KNN — Basic Implementation & k-Effect
# ============================================================

# Use the moon dataset again (good for visualizing boundaries)
X_knn, y_knn = make_moons(n_samples=300, noise=0.25, random_state=42)
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_knn, y_knn, test_size=0.3, random_state=42
)

# Try different k values
k_values = [1, 3, 5, 15, 30, 60]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for idx, k in enumerate(k_values):
    knn = KNeighborsClassifier(n_neighbors=k, weights='uniform', metric='euclidean')
    knn.fit(X_train_k, y_train_k)
    y_pred_k = knn.predict(X_test_k)
    acc = accuracy_score(y_test_k, y_pred_k)

    # Decision boundary
    h = 0.02
    x_min, x_max = X_knn[:, 0].min() - 0.5, X_knn[:, 0].max() + 0.5
    y_min, y_max = X_knn[:, 1].min() - 0.5, X_knn[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.coolwarm)
    axes[idx].scatter(X_train_k[:, 0], X_train_k[:, 1], c=y_train_k, cmap=plt.cm.coolwarm, edgecolors='k', s=30, alpha=0.7)
    axes[idx].set_title(f'k = {k}\nAccuracy: {acc:.3f}', fontsize=12, fontweight='bold')
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])

plt.suptitle('KNN Decision Boundaries: Effect of k', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== OBSERVATION ===")
print("k=1   → Very jagged boundary (overfitting, memorizes noise).")
print("k=5-15 → Smooth, reasonable boundary (good generalization).")
print("k=60  → Too smooth, ignores local structure (underfitting).")


In [ ]:
# ============================================================
# CELL 8: KNN — Distance Metrics & Weighting
# ============================================================

metrics = ['euclidean', 'manhattan', 'cosine']
weightings = ['uniform', 'distance']

fig, axes = plt.subplots(len(metrics), len(weightings), figsize=(12, 12))

for i, metric in enumerate(metrics):
    for j, weight in enumerate(weightings):
        knn_m = KNeighborsClassifier(n_neighbors=5, weights=weight, metric=metric)
        knn_m.fit(X_train_k, y_train_k)
        y_pred_m = knn_m.predict(X_test_k)
        acc = accuracy_score(y_test_k, y_pred_m)

        h = 0.02
        x_min, x_max = X_knn[:, 0].min() - 0.5, X_knn[:, 0].max() + 0.5
        y_min, y_max = X_knn[:, 1].min() - 0.5, X_knn[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        Z = knn_m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        axes[i, j].contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.coolwarm)
        axes[i, j].scatter(X_train_k[:, 0], X_train_k[:, 1], c=y_train_k, cmap=plt.cm.coolwarm, edgecolors='k', s=20, alpha=0.6)
        axes[i, j].set_title(f'{metric} | {weight}\nAcc: {acc:.3f}', fontsize=11)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

# Row labels
for i, metric in enumerate(metrics):
    axes[i, 0].set_ylabel(metric, fontsize=12, fontweight='bold', rotation=90, labelpad=20)

# Column labels
for j, weight in enumerate(weightings):
    axes[0, j].set_title(f'Weights: {weight}', fontsize=13, fontweight='bold', pad=20)

plt.suptitle('KNN: Distance Metrics vs. Weighting Schemes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== KEY INSIGHTS ===")
print("- Euclidean: Standard straight-line distance.")
print("- Manhattan: Grid-based distance (good for high-dimensional data).")
print("- Cosine: Angle between vectors (excellent for text/recommendations).")
print("- Uniform weighting: All neighbors vote equally.")
print("- Distance weighting: Closer neighbors have more influence.")


In [ ]:
# ============================================================
# CELL 9: KNN — The Curse of Dimensionality
# ============================================================

dimensions = [2, 5, 10, 20, 50, 100, 200, 500]
results = []

for d in dimensions:
    # Generate random data in d dimensions
    X_dim, y_dim = make_classification(
        n_samples=500, n_features=d, n_informative=d//2,
        n_redundant=0, random_state=42
    )
    X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
        X_dim, y_dim, test_size=0.3, random_state=42
    )

    # Scale features
    scaler_d = StandardScaler()
    X_train_d = scaler_d.fit_transform(X_train_d)
    X_test_d = scaler_d.transform(X_test_d)

    knn_d = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    knn_d.fit(X_train_d, y_train_d)
    acc = accuracy_score(y_test_d, knn_d.predict(X_test_d))
    results.append((d, acc))

df_dim = pd.DataFrame(results, columns=['Dimensions', 'Accuracy'])

plt.figure(figsize=(10, 5))
plt.plot(df_dim['Dimensions'], df_dim['Accuracy'], marker='o', linewidth=2, markersize=8, color='crimson')
plt.axhline(y=0.5, color='gray', linestyle='--', label='Random Guessing (50%)')
plt.xscale('log')
plt.xlabel('Number of Dimensions (log scale)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN Performance vs. Dimensionality\n(The Curse of Dimensionality)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(df_dim.to_string(index=False))

print("\n=== THE CURSE OF DIMENSIONALITY ===")
print("As dimensions increase, all points become roughly equidistant.")
print("KNN cannot distinguish near from far neighbors, so accuracy drops.")
print("SOLUTION: Use dimensionality reduction (PCA) or cosine distance for sparse data.")


## Part 4: Head-to-Head Comparison

Now let's compare all three algorithms on the same dataset using cross-validation.


In [ ]:
# ============================================================
# CELL 10: Algorithm Showdown — Cross-Validation Comparison
# ============================================================

# Use a more challenging dataset
X_comp, y_comp = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=1,
    class_sep=0.8,
    random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_comp, y_comp, test_size=0.2, random_state=42
)

scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_train_c)
X_test_c = scaler_c.transform(X_test_c)

# Define models
models = {
    'Naive Bayes (Gaussian)': GaussianNB(),
    'SVM (Linear)': SVC(kernel='linear', C=1.0, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'KNN (k=5, Euclidean)': KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'KNN (k=5, Cosine)': KNeighborsClassifier(n_neighbors=5, metric='cosine'),
}

# Evaluate
comparison = []
for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_c, y_train_c, cv=5, scoring='accuracy')

    # Test set performance
    model.fit(X_train_c, y_train_c)
    y_pred_c = model.predict(X_test_c)

    comparison.append({
        'Model': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': accuracy_score(y_test_c, y_pred_c),
        'Test F1': f1_score(y_test_c, y_pred_c),
    })

df_comp = pd.DataFrame(comparison)
print("=== ALGORITHM COMPARISON ===")
print(df_comp.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
x_pos = np.arange(len(df_comp))
axes[0].bar(x_pos - 0.2, df_comp['CV Mean'], 0.4, label='CV Mean', yerr=df_comp['CV Std'], capsize=5, color='steelblue')
axes[0].bar(x_pos + 0.2, df_comp['Test Accuracy'], 0.4, label='Test Accuracy', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_comp['Model'], rotation=45, ha='right')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Cross-Validation vs. Test Accuracy', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)

# F1 Score comparison
axes[1].barh(df_comp['Model'], df_comp['Test F1'], color='seagreen')
axes[1].set_xlabel('F1 Score')
axes[1].set_title('F1 Score Comparison', fontweight='bold')
axes[1].set_xlim(0, 1)
for i, v in enumerate(df_comp['Test F1']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()
